# Day 24 — Streaming responses

A non-streaming call returns nothing until the whole response is done — seconds of silence,
and a real risk of an HTTP timeout on long outputs. **Streaming** delivers the response as a
sequence of server-sent events as it's generated. Today: the event protocol, how to accumulate
text and tool-call JSON from partial deltas, the SDK helpers, and a working console streamer.

Runnable against a mock event generator; real SDK code alongside.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | Why stream | 4 min |
| 1 | The event sequence | 12 min |
| 2 | Accumulate text → a console typewriter | 12 min |
| 3 | Accumulate tool-call JSON from `input_json_delta` | 12 min |
| 4 | The SDK helpers: `text_stream`, `get_final_message` | 10 min |
| 5 | Tradeoffs: retries, errors, when not to stream | 7 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import json, time, sys
print("ready")

ready


## 0 — Why stream (4 min)

- **Time to first token (TTFT).** A user sees output start in ~0.5s instead of waiting ~10s
  for the whole answer. Perceived latency drops even though total time is the same.
- **HTTP timeouts.** Large `max_tokens` (32k–128k) on a non-streaming request can exceed the
  SDK's request timeout. The SDKs *require* streaming for very large `max_tokens`.
- **Incremental UI.** Typewriter effect, live tool-call display, early cancellation.
- **Progress on long agent turns.** You can log/act on partial output.

Cost: harder error handling (a failure can happen mid-stream, after you've shown text), and
you can't cleanly retry a half-delivered response.

## 1 — The event sequence (12 min)

A streamed response is a sequence of typed SSE events:

```
message_start          -> the Message shell: id, model, role, empty content, initial usage
content_block_start    -> a new block begins (index, type: text | tool_use | thinking)
content_block_delta    -> a piece of that block:
                            text_delta        {"text": "Hel"}
                            input_json_delta  {"partial_json": "{\"ci"}   (tool args, streamed!)
                            thinking_delta     {"thinking": "..."}
content_block_stop     -> that block is complete
  (start/delta*/stop repeats per block)
message_delta          -> top-level updates: stop_reason, and cumulative usage.output_tokens
message_stop           -> done
```

Let's build a mock stream and walk it.

In [2]:
def mock_stream(text="Hello! It is 14 degrees and clear in Paris.", tool_call=None, chunk=6):
    yield {"type": "message_start",
           "message": {"id": "msg_1", "model": "claude-opus-5", "role": "assistant",
                       "content": [], "usage": {"input_tokens": 812, "output_tokens": 0}}}
    idx = 0
    if text:
        yield {"type": "content_block_start", "index": idx, "content_block": {"type": "text", "text": ""}}
        for i in range(0, len(text), chunk):
            yield {"type": "content_block_delta", "index": idx,
                   "delta": {"type": "text_delta", "text": text[i:i+chunk]}}
        yield {"type": "content_block_stop", "index": idx}
        idx += 1
    if tool_call:
        blob = json.dumps(tool_call["input"])
        yield {"type": "content_block_start", "index": idx,
               "content_block": {"type": "tool_use", "id": "toolu_1", "name": tool_call["name"], "input": {}}}
        for i in range(0, len(blob), 5):
            yield {"type": "content_block_delta", "index": idx,
                   "delta": {"type": "input_json_delta", "partial_json": blob[i:i+5]}}
        yield {"type": "content_block_stop", "index": idx}
        idx += 1
    stop = "tool_use" if tool_call else "end_turn"
    yield {"type": "message_delta", "delta": {"stop_reason": stop}, "usage": {"output_tokens": 18}}
    yield {"type": "message_stop"}

for ev in mock_stream("Hi there.", chunk=3):
    print(ev)

{'type': 'message_start', 'message': {'id': 'msg_1', 'model': 'claude-opus-5', 'role': 'assistant', 'content': [], 'usage': {'input_tokens': 812, 'output_tokens': 0}}}
{'type': 'content_block_start', 'index': 0, 'content_block': {'type': 'text', 'text': ''}}
{'type': 'content_block_delta', 'index': 0, 'delta': {'type': 'text_delta', 'text': 'Hi '}}
{'type': 'content_block_delta', 'index': 0, 'delta': {'type': 'text_delta', 'text': 'the'}}
{'type': 'content_block_delta', 'index': 0, 'delta': {'type': 'text_delta', 'text': 're.'}}
{'type': 'content_block_stop', 'index': 0}
{'type': 'message_delta', 'delta': {'stop_reason': 'end_turn'}, 'usage': {'output_tokens': 18}}
{'type': 'message_stop'}


## 2 — Accumulate text → a console typewriter (12 min)

The common case: you just want the text as it arrives. Filter for `text_delta` and print.

In [3]:
def stream_text_to_console(events, delay=0.0):
    printed = []
    for ev in events:
        if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "text_delta":
            piece = ev["delta"]["text"]
            printed.append(piece)
            sys.stdout.write(piece); sys.stdout.flush()
            if delay: time.sleep(delay)
        elif ev["type"] == "message_delta":
            stop = ev["delta"].get("stop_reason")
        elif ev["type"] == "message_stop":
            sys.stdout.write("\n")
    return "".join(printed)

full = stream_text_to_console(mock_stream("Refunds are issued within 5 business days.", chunk=4))
print("reassembled:", repr(full))

Refu

nds 

are 

issu

ed w

...

busi

ness

 day

s.


reassembled: 'Refunds are issued within 5 business days.'


In [4]:
# A fuller reader: rebuild the complete Message from events (what get_final_message does).
def reduce_stream(events):
    msg = None; blocks = {}
    for ev in events:
        t = ev["type"]
        if t == "message_start":
            msg = dict(ev["message"])
        elif t == "content_block_start":
            blocks[ev["index"]] = dict(ev["content_block"])
            if blocks[ev["index"]]["type"] == "tool_use":
                blocks[ev["index"]]["_json"] = ""
        elif t == "content_block_delta":
            d = ev["delta"]; b = blocks[ev["index"]]
            if d["type"] == "text_delta": b["text"] += d["text"]
            elif d["type"] == "input_json_delta": b["_json"] += d["partial_json"]
            elif d["type"] == "thinking_delta": b.setdefault("thinking", ""); b["thinking"] += d["thinking"]
        elif t == "content_block_stop":
            b = blocks[ev["index"]]
            if b["type"] == "tool_use":
                b["input"] = json.loads(b.pop("_json") or "{}")
        elif t == "message_delta":
            msg["stop_reason"] = ev["delta"].get("stop_reason")
            msg["usage"].update(ev.get("usage", {}))
    msg["content"] = [blocks[i] for i in sorted(blocks)]
    return msg

m = reduce_stream(mock_stream("Two degrees.", tool_call={"name": "get_weather", "input": {"city": "Oslo"}}))
print(json.dumps(m, indent=1))

{
 "id": "msg_1",
 "model": "claude-opus-5",
 "role": "assistant",
 "content": [
  {
   "type": "text",
   "text": "Two degrees."
  },
  {
   "type": "tool_use",
   "id": "toolu_1",
   "name": "get_weather",
   "input": {
    "city": "Oslo"
   }
  }
 ],
 "usage": {
  "input_tokens": 812,
  "output_tokens": 18
 },
 "stop_reason": "tool_use"
}


## 3 — Accumulate tool-call JSON (12 min)

Tool arguments stream too, as `input_json_delta` fragments of a JSON string. You **cannot
`json.loads` a fragment** — you concatenate `partial_json` across all deltas for that block,
then parse once at `content_block_stop`.

In [5]:
def stream_with_tools(events, on_text=None, on_tool=None):
    buffers = {}   # index -> {"kind", "name", "json" or "text"}
    for ev in events:
        t = ev["type"]
        if t == "content_block_start":
            cb = ev["content_block"]
            buffers[ev["index"]] = ({"kind": "tool", "name": cb["name"], "json": ""}
                                    if cb["type"] == "tool_use" else {"kind": "text", "text": ""})
        elif t == "content_block_delta":
            d = ev["delta"]; buf = buffers[ev["index"]]
            if d["type"] == "text_delta":
                buf["text"] += d["text"]
                if on_text: on_text(d["text"])
            elif d["type"] == "input_json_delta":
                buf["json"] += d["partial_json"]        # accumulate — do NOT parse yet
        elif t == "content_block_stop":
            buf = buffers[ev["index"]]
            if buf["kind"] == "tool":
                args = json.loads(buf["json"] or "{}")
                if on_tool: on_tool(buf["name"], args)
        elif t == "message_delta":
            stop = ev["delta"].get("stop_reason")
    return stop

calls = []
stop = stream_with_tools(
    mock_stream("Let me check.", tool_call={"name": "get_weather", "input": {"city": "Paris", "units": "celsius"}}),
    on_text=lambda s: print("TEXT:", repr(s)),
    on_tool=lambda n, a: calls.append((n, a)))
print("\nstop_reason:", stop, "| tool calls:", calls)

TEXT: 'Let me'
TEXT: ' check'
TEXT: '.'

stop_reason: tool_use | tool calls: [('get_weather', {'city': 'Paris', 'units': 'celsius'})]


Why stream tool JSON at all? For a UI that shows *"searching the knowledge base for
'refund policy'…"* the instant the model commits to the call — before it finishes writing the
arguments. If you don't need that, `stream.get_final_message()` hands you the parsed `input`
and you never touch `partial_json`.

**`eager_input_streaming: true`** on a tool definition makes the API emit `input_json_delta`
sooner (before the model has finished validating the JSON) — not a beta, just a flag.

## 4 — The SDK helpers (10 min)

```python
with client.messages.stream(
    model="claude-opus-5", max_tokens=1024,
    messages=[{"role": "user", "content": "Explain streaming"}],
) as stream:
    for text in stream.text_stream:          # yields only text pieces, already filtered
        print(text, end="", flush=True)
    final = stream.get_final_message()        # the complete Message, after the loop

print(final.stop_reason, final.usage.output_tokens)
```

- `stream.text_stream` — the 90% case: an iterator of text chunks, no event parsing.
- `for event in stream:` — raw events, when you need tool-JSON-as-it-arrives or thinking deltas.
- `stream.get_final_message()` — the fully reassembled `Message`. Always available after the
  stream is consumed.
- For large `max_tokens` (128k), streaming + `get_final_message()` is the *only* safe way —
  a non-streaming call would hit the HTTP timeout.

**Don't** rebuild these with `.on()` callbacks wrapped in a `Promise` / manual accumulation —
the helper already does it correctly (including edge cases like `message_delta` usage).

In [6]:
# our text_stream equivalent, as a generator
def text_stream(events):
    for ev in events:
        if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "text_delta":
            yield ev["delta"]["text"]

evs = list(mock_stream("The answer streams in piece by piece, which feels faster.", chunk=8))
out = ""
for piece in text_stream(evs):
    out += piece
print("streamed:", out)

streamed: The answer streams in piece by piece, which feels faster.


## 5 — Tradeoffs (7 min)

| | Non-streaming | Streaming |
| --- | --- | --- |
| TTFT | = total time | ~0.5s |
| Perceived speed | slow | fast |
| Large `max_tokens` | may hit HTTP timeout | required |
| Retry on failure | clean (nothing shown yet) | messy — you may have already rendered partial text |
| Error surface | one exception at the end | can fail mid-stream after `message_start` |
| Token/stop info | in the response object | in `message_delta` near the end / `get_final_message()` |
| Code complexity | low | medium (or low with `text_stream`) |

**Mid-stream failures:** the connection can drop after some deltas. Handle it: catch the
exception around the stream loop, decide whether to show the partial with a "…(interrupted)"
marker or discard it, and retry as a *fresh* request (you can't resume a stream). Servers may
also send an `error` event.

**When not to stream:** batch/offline jobs, when you need the whole answer before doing
anything with it (parsing JSON, routing), classification with tiny outputs, or inside a tool
loop where you just want the final message — `create()` is simpler there.

In [7]:
def robust_stream(events, on_text):
    acc = []
    try:
        for ev in events:
            if ev["type"] == "error":
                raise RuntimeError(ev["error"]["message"])
            if ev["type"] == "content_block_delta" and ev["delta"]["type"] == "text_delta":
                acc.append(ev["delta"]["text"]); on_text(ev["delta"]["text"])
    except (RuntimeError, ConnectionError) as e:
        return "".join(acc) + "  …(interrupted: %s)" % e, False
    return "".join(acc), True

def flaky_stream():
    yield {"type": "message_start", "message": {}}
    yield {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": ""}}
    for w in ["The ", "refund ", "process ", "takes "]:
        yield {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": w}}
    yield {"type": "error", "error": {"type": "overloaded_error", "message": "server overloaded"}}

text, ok = robust_stream(flaky_stream(), on_text=lambda s: None)
print(f"ok={ok}  ->  {text!r}")
print("-> retry as a NEW request; you cannot resume the stream.")

ok=False  ->  'The refund process takes   …(interrupted: server overloaded)'
-> retry as a NEW request; you cannot resume the stream.


## 6 — Exercises

1. **Word-wrap typewriter.** Extend `stream_text_to_console` to wrap output at 72 columns
   without breaking words, buffering the current word until a space arrives.
2. **TTFT vs total.** Add timestamps to `mock_stream` (sleep between chunks). Measure time to
   the first `text_delta` vs time to `message_stop`. Vary `chunk` size.
3. **Interleaved thinking + text.** Make `mock_stream` emit a `thinking` block (with
   `thinking_delta`s) before the text block. Render thinking dimmed / in brackets and text
   normally; show `reduce_stream` keeps them as separate content blocks.
4. **Two tool calls in one stream.** Emit two `tool_use` blocks (indices 1 and 2) with
   interleaved `input_json_delta`s. Show your accumulator keys buffers by `index` so the two
   JSON strings don't get mixed.
5. **Partial-JSON trap.** Try to `json.loads` after every `input_json_delta` instead of at
   `content_block_stop`. Show it raises `JSONDecodeError` until the last fragment, and that
   only the accumulate-then-parse approach works.
6. **Cancellation.** Add a `stop_after_tokens` arg to `text_stream` that stops iterating once
   N tokens have arrived (simulating a user hitting "stop"). What happens to `usage` and the
   billing? (You're still billed for what the server generated before you disconnected.)

In [8]:
# ---- Solution 4 ----
def two_tool_stream():
    yield {"type": "message_start", "message": {}}
    yield {"type": "content_block_start", "index": 0, "content_block": {"type": "tool_use", "id": "t1", "name": "get_weather", "input": {}}}
    yield {"type": "content_block_start", "index": 1, "content_block": {"type": "tool_use", "id": "t2", "name": "convert", "input": {}}}
    a, b = json.dumps({"city": "Paris"}), json.dumps({"amount": 50})
    for i in range(0, max(len(a), len(b)), 4):
        if i < len(a): yield {"type": "content_block_delta", "index": 0, "delta": {"type": "input_json_delta", "partial_json": a[i:i+4]}}
        if i < len(b): yield {"type": "content_block_delta", "index": 1, "delta": {"type": "input_json_delta", "partial_json": b[i:i+4]}}
    yield {"type": "content_block_stop", "index": 0}
    yield {"type": "content_block_stop", "index": 1}
    yield {"type": "message_delta", "delta": {"stop_reason": "tool_use"}}
    yield {"type": "message_stop"}

bufs = {}
for ev in two_tool_stream():
    if ev["type"] == "content_block_start" and ev["content_block"]["type"] == "tool_use":
        bufs[ev["index"]] = {"name": ev["content_block"]["name"], "json": ""}
    elif ev["type"] == "content_block_delta" and ev["delta"]["type"] == "input_json_delta":
        bufs[ev["index"]]["json"] += ev["delta"]["partial_json"]
    elif ev["type"] == "content_block_stop" and ev["index"] in bufs:
        bufs[ev["index"]]["args"] = json.loads(bufs[ev["index"]]["json"])
print("S4:", [(b["name"], b["args"]) for b in bufs.values()])

S4: [('get_weather', {'city': 'Paris'}), ('convert', {'amount': 50})]


In [9]:
# ---- Solution 5 ----
frag = ""
errors_until_end = 0
blob = json.dumps({"city": "Paris", "units": "celsius"})
for i in range(0, len(blob), 5):
    frag += blob[i:i+5]
    try:
        json.loads(frag)
    except json.JSONDecodeError:
        errors_until_end += 1
print(f"S5: {errors_until_end} JSONDecodeErrors while fragments were partial; "
      f"parse succeeds only once frag == the complete string.")

S5: 7 JSONDecodeErrors while fragments were partial; parse succeeds only once frag == the complete string.


### Solutions 1, 2, 3, 6 (sketch)

**S1:** keep a `word` buffer; on each `text_delta`, split on spaces — append complete words to
a `line`, flush `line` + newline when `len(line) + len(word) > 72`, keep the trailing partial
word in `word`.

**S2:** `first_token_t - start_t` is your TTFT; `stop_t - start_t` is total. Smaller `chunk`
(more, smaller deltas) barely changes TTFT but adds per-event overhead; TTFT is dominated by
the model's prefill, not by chunking.

**S3:** emit `content_block_start {type:"thinking"}` → `thinking_delta`s →
`content_block_stop`, then the text block at index 1. `reduce_stream` already handles
`thinking_delta`; the final `content` has `[{type:"thinking",...}, {type:"text",...}]`. Render
thinking with a dim style and don't send it to the user as the answer.

**S6:** break out of the `for piece in text_stream` loop early. The HTTP connection closes,
but the server had already generated some or all of the response — **you're billed for the
output tokens the server produced**, not for what you read. `get_final_message()` won't have
the full text if you stopped early; check `usage` from the last `message_delta` you saw.

## Self-check quiz

1. Give two concrete reasons to stream rather than use a plain `create()` call.
2. List the event types in order for a response with one text block.
3. Why can't you `json.loads` an `input_json_delta`'s `partial_json`?
4. What does `stream.text_stream` give you vs iterating raw events?
5. A stream fails after emitting half the text. What are your options, and can you resume?
6. When is streaming the *wrong* choice?
7. You cancel a stream early. What are you billed for?

### Answer key

1. Any two: lower time-to-first-token / better perceived latency; avoiding HTTP timeouts on
   large `max_tokens` (streaming is required there); incremental UI (typewriter, live tool
   display); acting on partial output during long agent turns.
2. `message_start` → `content_block_start` (text) → `content_block_delta` (text_delta) × N →
   `content_block_stop` → `message_delta` (stop_reason, usage) → `message_stop`.
3. It's a fragment of a JSON string, not valid JSON on its own. You concatenate all
   `partial_json` pieces for that block and parse once at `content_block_stop`.
4. `text_stream` yields only the text pieces, already filtered and accumulated for you — the
   simple path. Raw events give you tool-call JSON deltas, thinking deltas, block boundaries,
   and usage — needed for live tool-call UIs or thinking display.
5. Catch the exception around the loop; show the accumulated partial with an "(interrupted)"
   marker or discard it; retry as a **fresh** request. You cannot resume a stream.
6. Batch/offline jobs; when you need the complete output before acting on it (JSON parsing,
   routing); tiny classification outputs; inside a tool loop where you only want the final
   message.
7. The output tokens the server already generated before the connection closed — not just
   what you received. Streaming doesn't reduce token cost, only latency.

## Where this goes next

Week 8 done — you know the SDK, tool use, and streaming end to end. **Week 9 — Evals &
Observability**: what makes a good eval, building an eval harness with no ground-truth labels,
and tracing an LLM pipeline (Day 25).